# Отчёт по лабораторной работе №1

## Введение в Spring Boot

**Дисциплина:** Разработка кроссплатформенных приложений, семестр 2

**Выполнил:** студент группы 12002408

Блинов Кирилл Игоревич

**Проверил:** ассистент кафедры информационных и робототехнических систем

Пузырев Никита Сергеевич

## Цель работы

Получить практические навыки работы со Spring Boot, создать веб-приложение
для хранения заметок и изучить обработку HTTP-запросов.

## Задание

1. Изучить теоретический материал.
2. Разработать приложение для добавления, просмотра, изменения и удаления сообщений.
3. Реализовать два дополнительных метода: очистку всех сообщений и подсчёт их количества.
4. Проверить работу приложения и подготовить отчёт.

## Ход работы

### 1. Создание проекта

Создан Gradle-проект `Demo`. Использованы Java 17, Gradle 8.12 и Spring Boot 3.2.3. Для веб-приложения подключён spring-boot-starter-web.

### 2. Настройка Gradle

В `build.gradle` указаны плагины, версия Java и зависимости.
Плагин Spring Boot позволяет запускать приложение командой `bootRun`
и собирать исполняемый JAR командой `bootJar`.

```groovy
plugins {
    id 'java'
    id 'org.springframework.boot' version '3.2.3'
    id 'io.spring.dependency-management' version '1.1.4'
}

group = 'ru.bsuedu.cad.lab'
version = '1.0'

java {
    sourceCompatibility = JavaVersion.VERSION_17
    targetCompatibility = JavaVersion.VERSION_17
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
}

tasks.withType(JavaCompile).configureEach {
    options.encoding = 'UTF-8'
    options.compilerArgs.add('-parameters')
}
```

### 3. Точка входа приложения

Класс `DemoApplication` запускает приложение.
Аннотация `@SpringBootApplication` включает автоматическую настройку
Spring Boot и поиск компонентов.

```java
package ru.bsuedu.cad.lab;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class DemoApplication {

    public static void main(String[] args) {
        SpringApplication.run(DemoApplication.class, args);
    }
}
```

### 4. Контроллер сообщений

В `MessageController` создан список `userMessages` для хранения заметок.
Аннотация `@RestController` обозначает обработчик HTTP-запросов.
`@RequestBody` получает текст сообщения из тела запроса,
а `@PathVariable` получает индекс из адреса.

```java
package ru.bsuedu.cad.lab;

import org.springframework.web.bind.annotation.DeleteMapping;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PathVariable;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PutMapping;
import org.springframework.web.bind.annotation.RequestBody;
import org.springframework.web.bind.annotation.RestController;

import java.util.ArrayList;
import java.util.List;

@RestController
public class MessageController {

    private final List<String> userMessages = new ArrayList<>();

    @GetMapping("/")
    public String helloWorld() {
        return "привет мир";
    }

    @GetMapping("/messages")
    public synchronized List<String> getAllMessages() {
        return new ArrayList<>(userMessages);
    }

    @PostMapping("/messages")
    public synchronized String publishMessage(@RequestBody String message) {
        userMessages.add(message);
        return "сообщение добавлено";
    }

    @PutMapping("/messages/{index}")
    public synchronized String updateMessage(@PathVariable int index,
                                             @RequestBody String message) {
        if (index >= 0 && index < userMessages.size()) {
            userMessages.set(index, message);
            return "сообщение изменено";
        }
        return "сообщение с индексом " + index + " не найдено";
    }

    @DeleteMapping("/messages/{index}")
    public synchronized String deleteMessage(@PathVariable int index) {
        if (index >= 0 && index < userMessages.size()) {
            userMessages.remove(index);
            return "сообщение удалено";
        }
        return "сообщение с индексом " + index + " не найдено";
    }

    @DeleteMapping("/messages")
    public synchronized String clearAllMessages() {
        userMessages.clear();
        return "все сообщения удалены";
    }

    @GetMapping("/messages/count")
    public synchronized int countMessages() {
        return userMessages.size();
    }
}
```

### 5. Запросы приложения

| Метод | Адрес | Назначение |
| --- | --- | --- |
| GET | `/` | Приветствие |
| GET | `/messages` | Получить список сообщений |
| POST | `/messages` | Добавить сообщение |
| PUT | `/messages/{index}` | Изменить сообщение по индексу |
| DELETE | `/messages/{index}` | Удалить сообщение по индексу |
| DELETE | `/messages` | Очистить все сообщения |
| GET | `/messages/count` | Получить количество сообщений |

### 6. Запуск приложения

При обращении к корневому адресу сервер возвращает привет мир. Обычный запуск через bootRun использует порт 8080.

### 7. Проверка основных операций

Запросы отправлены в Insomnia к приложению на `localhost:8080`.
Проверены приветствие, добавление и получение сообщения, изменение текста
по индексу 0 и удаление сообщения по индексу 0. Сервер вернул успешные ответы.

**а) GET /: приветствие**

[![GET /: приветствие](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-greeting.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-greeting.png)

**б) POST /messages: добавление сообщения**

[![POST /messages: добавление сообщения](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-post-message.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-post-message.png)

**в) GET /messages: получение списка**

[![GET /messages: получение списка](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-get-messages.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-get-messages.png)

**г) PUT /messages/0: изменение сообщения**

[![PUT /messages/0: изменение сообщения](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-put-message.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-put-message.png)

**д) DELETE /messages/0: удаление сообщения**

[![DELETE /messages/0: удаление сообщения](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-delete-message.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-delete-message.png)

### 8. Дополнительные задания

**Пункт 1. Очистка списка**

Метод `clearAllMessages()` удаляет все сообщения. Для него используется запрос
`DELETE /messages`.

```java
@DeleteMapping("/messages")
public synchronized String clearAllMessages() {
    userMessages.clear();
    return "все сообщения удалены";
}
```

**Пункт 6. Подсчёт сообщений**

Метод `countMessages()` возвращает размер списка. Для него используется запрос
`GET /messages/count`.

```java
@GetMapping("/messages/count")
public synchronized int countMessages() {
    return userMessages.size();
}
```

В Insomnia проверены два дополнительных метода. Запрос `GET /messages/count`
вернул количество сообщений: 3. Для очистки выполнен `DELETE /messages`.
Отдельный запрос `GET /messages` показывает пустой список.

**а) GET /messages/count: количество сообщений**

[![GET /messages/count: количество сообщений](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-count-messages.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-count-messages.png)

**б) DELETE /messages: очистка списка**

[![DELETE /messages: очистка списка](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-clear-messages.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-clear-messages.png)

**в) GET /messages: пустой список**

[![GET /messages: пустой список](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-empty-messages.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-empty-messages.png)

### 9. Проверка неверного индекса

В Insomnia отправлены запросы `PUT /messages/-1` и `DELETE /messages/100`.
В обоих случаях сервер вернул сообщение об отсутствии заметки с указанным индексом.
Код ответа 200 сохранён по примеру из методички.

**а) PUT /messages/-1: отрицательный индекс**

[![PUT /messages/-1: отрицательный индекс](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-invalid-put.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-invalid-put.png)

**б) DELETE /messages/100: индекс за пределами списка**

[![DELETE /messages/100: индекс за пределами списка](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-invalid-delete.png)](https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/insomnia-invalid-delete.png)

## Вывод

В ходе лабораторной работы создано веб-приложение Spring Boot для хранения заметок. Реализованы добавление, просмотр, изменение и удаление сообщений. Выполнены дополнительные задания по очистке списка и подсчёту сообщений. Работа приложения проверена HTTP-запросами.